# Confidence Regression: Audio/Text Embedding + 23 Classes

베이스라인 분류 코드는 쓰지 않고, `BSD10k_metadata.csv`의 정수형 `confidence`를 `float32` 회귀 타깃으로 바꿔 예측합니다.

- 입력: CLAP audio embedding + CLAP text embedding + 23개 class one-hot
- 타깃: `confidence` float 값
- 지표: rounded accuracy, MAE, MSE
- accuracy 정의: 회귀 예측값을 confidence 범위로 clip한 뒤 반올림해서 원래 정수 label과 비교

In [ ]:
from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

In [ ]:
# ============================================================
# 설정
# ============================================================
SEED = 42
K_FOLDS = 5
VAL_SIZE = 0.2

BATCH_SIZE = 256
NUM_EPOCHS = 80
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 12
DROPOUT = 0.25

# 입력 feature 사용 여부
USE_AUDIO_EMB = True
USE_TEXT_EMB = True
USE_CLASS_ONEHOT = True

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

METADATA_PATH = PROJECT_ROOT / 'data' / 'metadata' / 'BSD10k_metadata.csv'
AUDIO_EMB_DIR = PROJECT_ROOT / 'data' / 'features' / 'clap_audio_embeddings'
TEXT_EMB_DIR = PROJECT_ROOT / 'data' / 'features' / 'clap_text_embeddings'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'confidence_regression_embeddings_class'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PROJECT_ROOT:', PROJECT_ROOT)
print('device:', device)

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_embedding(path: Path) -> np.ndarray:
    emb = np.load(path).astype(np.float32)
    return emb.reshape(-1)


def rounded_confidence_accuracy(y_true_float, y_pred_float):
    y_true_int = np.rint(y_true_float).astype(int)
    min_label = int(np.min(y_true_int))
    max_label = int(np.max(y_true_int))
    y_pred_int = np.rint(np.clip(y_pred_float, min_label, max_label)).astype(int)
    return accuracy_score(y_true_int, y_pred_int)


set_seed(SEED)

In [ ]:
# ============================================================
# 메타데이터 로드 및 feature 경로 구성
# ============================================================
df = pd.read_csv(METADATA_PATH)

# confidence는 원본이 정수형 메타데이터지만 회귀 타깃으로 float32 사용
df = df.dropna(subset=['confidence', 'class']).copy()
df['confidence_float'] = df['confidence'].astype(np.float32)
df['confidence_int'] = df['confidence'].astype(int)
df['sound_id'] = df['sound_id'].astype(str)
df['audio_emb_path'] = df['sound_id'].map(lambda x: AUDIO_EMB_DIR / f'{x}.npy')
df['text_emb_path'] = df['sound_id'].map(lambda x: TEXT_EMB_DIR / f'{x}.npy')

if USE_AUDIO_EMB:
    df = df[df['audio_emb_path'].map(lambda p: p.exists())].copy()
if USE_TEXT_EMB:
    df = df[df['text_emb_path'].map(lambda p: p.exists())].copy()

class_names = sorted(df['class'].unique().tolist())
assert len(class_names) == 23, f'Expected 23 classes, got {len(class_names)}'
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
df['class_23_idx'] = df['class'].map(class_to_idx).astype(int)

print('rows:', len(df))
print('classes:', len(class_names))
print('confidence labels:', sorted(df['confidence_int'].unique().tolist()))
display(df[['sound_id', 'class', 'class_23_idx', 'confidence', 'confidence_float']].head())

In [ ]:
# ============================================================
# Feature matrix 생성
# ============================================================
def build_feature_matrix(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    parts = []

    if USE_AUDIO_EMB:
        audio = np.stack([load_embedding(p) for p in frame['audio_emb_path']], axis=0)
        parts.append(audio)
        print('audio:', audio.shape)

    if USE_TEXT_EMB:
        text = np.stack([load_embedding(p) for p in frame['text_emb_path']], axis=0)
        parts.append(text)
        print('text:', text.shape)

    if USE_CLASS_ONEHOT:
        class_onehot = np.eye(23, dtype=np.float32)[frame['class_23_idx'].to_numpy()]
        parts.append(class_onehot)
        print('class_onehot:', class_onehot.shape)

    X = np.concatenate(parts, axis=1).astype(np.float32)
    y_float = frame['confidence_float'].to_numpy(dtype=np.float32)
    y_int = frame['confidence_int'].to_numpy(dtype=np.int64)
    return X, y_float, y_int


X, y_float, y_int = build_feature_matrix(df)
print('X:', X.shape, 'y:', y_float.shape)

In [ ]:
class ConfidenceRegressor(nn.Module):
    def __init__(self, input_dim: int, dropout: float = 0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


def make_loader(X_np, y_np, batch_size=BATCH_SIZE, shuffle=False):
    X_tensor = torch.from_numpy(X_np.astype(np.float32))
    y_tensor = torch.from_numpy(y_np.astype(np.float32))
    dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

In [ ]:
@torch.no_grad()
def predict_model(model, loader, device):
    model.eval()
    preds = []
    targets = []
    for xb, yb in loader:
        xb = xb.to(device)
        pred = model(xb).detach().cpu().numpy()
        preds.append(pred)
        targets.append(yb.numpy())
    return np.concatenate(targets), np.concatenate(preds)


def compute_metrics(y_true, y_pred):
    return {
        'accuracy': rounded_confidence_accuracy(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'mse': mean_squared_error(y_true, y_pred),
    }


def train_one_fold(fold, X_train, y_train, X_val, y_val, input_dim):
    set_seed(SEED + fold)
    model = ConfidenceRegressor(input_dim=input_dim, dropout=DROPOUT).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=4
    )

    train_loader = make_loader(X_train, y_train, shuffle=True)
    val_loader = make_loader(X_val, y_val, shuffle=False)

    best_val_mae = float('inf')
    best_state = None
    bad_epochs = 0
    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        n_seen = 0

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            n_seen += xb.size(0)

        train_mse = running_loss / max(n_seen, 1)
        val_true, val_pred = predict_model(model, val_loader, device)
        val_metrics = compute_metrics(val_true, val_pred)
        scheduler.step(val_metrics['mae'])

        row = {
            'fold': fold,
            'epoch': epoch,
            'train_mse': train_mse,
            'val_accuracy': val_metrics['accuracy'],
            'val_mae': val_metrics['mae'],
            'val_mse': val_metrics['mse'],
            'lr': optimizer.param_groups[0]['lr'],
        }
        history.append(row)

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"fold {fold} epoch {epoch:03d} | "
                f"train_mse={train_mse:.4f} | "
                f"val_acc={val_metrics['accuracy']:.4f} | "
                f"val_mae={val_metrics['mae']:.4f} | "
                f"val_mse={val_metrics['mse']:.4f}"
            )

        if val_metrics['mae'] < best_val_mae:
            best_val_mae = val_metrics['mae']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f'fold {fold} early stopping at epoch {epoch}')
                break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

In [ ]:
# ============================================================
# 5-fold confidence regression
# ============================================================
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

fold_results = []
all_histories = []
all_predictions = []

for fold, (trainval_idx, test_idx) in enumerate(skf.split(X, y_int)):
    print('\n' + '=' * 70)
    print(f'FOLD {fold} / {K_FOLDS - 1}')
    print('=' * 70)

    trainval_y_int = y_int[trainval_idx]
    sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED + fold)
    train_rel, val_rel = next(sss.split(np.zeros(len(trainval_idx)), trainval_y_int))
    train_idx = trainval_idx[train_rel]
    val_idx = trainval_idx[val_rel]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx]).astype(np.float32)
    X_val = scaler.transform(X[val_idx]).astype(np.float32)
    X_test = scaler.transform(X[test_idx]).astype(np.float32)

    y_train = y_float[train_idx]
    y_val = y_float[val_idx]
    y_test = y_float[test_idx]

    print(f'train={len(train_idx)} val={len(val_idx)} test={len(test_idx)} input_dim={X_train.shape[1]}')

    start = time.time()
    model, history = train_one_fold(fold, X_train, y_train, X_val, y_val, X_train.shape[1])
    elapsed = time.time() - start
    all_histories.append(history)

    test_loader = make_loader(X_test, y_test, shuffle=False)
    test_true, test_pred = predict_model(model, test_loader, device)
    test_metrics = compute_metrics(test_true, test_pred)

    fold_dir = OUTPUT_DIR / f'fold_{fold}'
    fold_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), fold_dir / 'best_model.pth')
    history.to_csv(fold_dir / 'history.csv', index=False)

    pred_df = df.iloc[test_idx][['sound_id', 'class', 'class_23_idx', 'confidence_int']].copy()
    pred_df['confidence_true_float'] = test_true
    pred_df['confidence_pred_float'] = test_pred
    pred_df['confidence_pred_round'] = np.rint(
        np.clip(test_pred, y_int.min(), y_int.max())
    ).astype(int)
    pred_df['fold'] = fold
    pred_df.to_csv(fold_dir / 'test_predictions.csv', index=False)
    all_predictions.append(pred_df)

    result = {
        'fold': fold,
        'train_size': len(train_idx),
        'val_size': len(val_idx),
        'test_size': len(test_idx),
        'accuracy': test_metrics['accuracy'],
        'mae': test_metrics['mae'],
        'mse': test_metrics['mse'],
        'elapsed_sec': elapsed,
    }
    fold_results.append(result)

    print('\n-- Fold Results --')
    print(f"accuracy: {test_metrics['accuracy']:.4f}")
    print(f"mae:      {test_metrics['mae']:.4f}")
    print(f"mse:      {test_metrics['mse']:.4f}")

results_df = pd.DataFrame(fold_results)
predictions_df = pd.concat(all_predictions, ignore_index=True)
histories_df = pd.concat(all_histories, ignore_index=True)

results_df.to_csv(OUTPUT_DIR / 'fold_results.csv', index=False)
predictions_df.to_csv(OUTPUT_DIR / 'all_test_predictions.csv', index=False)
histories_df.to_csv(OUTPUT_DIR / 'all_histories.csv', index=False)

display(results_df)

In [ ]:
# ============================================================
# 최종 요약
# ============================================================
summary = results_df[['accuracy', 'mae', 'mse']].agg(['mean', 'std']).T
display(summary)

summary_dict = {
    metric: {
        'mean': float(summary.loc[metric, 'mean']),
        'std': float(summary.loc[metric, 'std']),
    }
    for metric in summary.index
}

with open(OUTPUT_DIR / 'summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary_dict, f, ensure_ascii=False, indent=2)

print('saved to:', OUTPUT_DIR)

In [ ]:
# 예측값 분포 확인
display(
    predictions_df[['confidence_true_float', 'confidence_pred_float', 'confidence_pred_round']]
    .describe()
)

display(
    pd.crosstab(
        predictions_df['confidence_int'],
        predictions_df['confidence_pred_round'],
        rownames=['true'],
        colnames=['pred_round'],
    )
)